In [ ]:
import os
import shutil
import pandas as pd
import gc
import json
from IPython.display import clear_output
from dotenv import load_dotenv
from groq import Groq
from agents.manager_agent import ManagerAgent
from llm_loader import LLMManager
import config

def get_safe_path(target_path):
    """Tìm đường dẫn file, nếu không thấy sẽ quét cục bộ và bỏ qua /content/drive"""
    if os.path.exists(target_path):
        return target_path
    
    filename = os.path.basename(target_path)
    print(f"⚠️ Không tìm thấy file tại đường dẫn mặc định. Đang truy tìm '{filename}'...")
    
    # Chỉ quét trong /content nhưng bỏ qua drive
    for root, dirs, files in os.walk('/content', topdown=True):
        if 'drive' in dirs:
            dirs.remove('drive')  # <--- ĐÂY LÀ DÒNG QUAN TRỌNG NHẤT: Bỏ qua thư mục drive
            
        if filename in files:
            found_path = os.path.join(root, filename)
            print(f"🎯 Đã tìm thấy file tại: {found_path}")
            return found_path
    return None

# ==========================================
# 🔐 BẢO MẬT: NẠP API KEY TỪ FILE .ENV
# ==========================================
load_dotenv() # Tự động tìm và load các biến từ file .env
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ LỖI: Chưa tìm thấy GROQ_API_KEY trong file .env. Hãy thêm nó vào trước khi chạy!")

# Khởi tạo Groq Client
groq_client = Groq(api_key=groq_api_key)

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN (DÙNG HÀM TÌM AN TOÀN)
# ==========================================
# 1. Đảm bảo đứng đúng thư mục gốc của Colab
if os.path.exists('/content'):
    os.chdir('/content')

RAW_EXCEL_PATH = "Thư mục KQ/VB gốc/variant B.xlsx"
EXCEL_FILE_PATH = get_safe_path(RAW_EXCEL_PATH)

if not EXCEL_FILE_PATH:
    raise FileNotFoundError(f"❌ LỖI: Không thể tìm thấy file '{os.path.basename(RAW_EXCEL_PATH)}' trong hệ thống (đã bỏ qua Drive).")

OUTPUT_EXCEL_PATH = "Thư mục KQ/VB sau tóm tắt/test_results.xlsx"
BATCH_BACKUP_DIR = "Thư mục KQ/Data_Results"
MAX_DOCS = 50

os.makedirs(BATCH_BACKUP_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_EXCEL_PATH), exist_ok=True)

df = pd.read_excel(EXCEL_FILE_PATH)
df = df.head(MAX_DOCS)

if 'Summary' not in df.columns:
    df['Summary'] = ""
if 'Result' not in df.columns:
    df['Result'] = ""

llm = LLMManager()

print(f"✅ Đã tải file Excel. Tổng số văn bản sẽ xử lý: {len(df)}")
print("🚀 Bắt đầu quá trình treo máy tự động...")

# ==========================================
# HÀM CHẤM ĐIỂM BẰNG GROQ API (PROMPT CHUYÊN SÂU)
# ==========================================
def evaluate_summary_with_groq(article, summary):
    prompt = f"""You are a strict Linguistics Expert and AI Evaluator, specializing in assessing the quality of Vietnamese text summaries.
Your task is to evaluate the summary below based on the original text. Please score from 0.0 (Worst) to 1.0 (Perfect) for the following 3 criteria:

1. Faithfulness: Does the summary hallucinate additional information, contain incorrect figures, or distort the meaning of the original text?
2. Relevance: Does the summary cover all the most important key points? Does it miss any core messages?
3. Coherence & Conciseness: Is the summary fluent, logical, easy to read, and successfully eliminate redundant details or repetitive words?

[ORIGINAL TEXT]
{article}

[AI SUMMARY]
{summary}

⚠️ STRICT OUTPUT REQUIREMENTS:
- You MUST return the result in JSON format. DO NOT output any additional explanations, markdown formatting (such as ```json), or greetings outside the JSON block.
- The "reason" fields MUST be written in Vietnamese. Provide sharp analysis and point out specific errors (if any) to explain why that specific score was given.
- The JSON structure must be exactly as follows:
{{
  "faithfulness_score": <float>,
  "faithfulness_reason": "<string>",
  "relevance_score": <float>,
  "relevance_reason": "<string>",
  "coherence_conciseness_score": <float>,
  "coherence_conciseness_reason": "<string>"
}}"""

    try:
        response = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a JSON-generating evaluation AI. Output ONLY valid JSON matching the user's schema."},
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile", # Dùng model 70B cực kỳ thông minh của Groq để làm trọng tài
            temperature=0.1,         # Nhiệt độ thấp để đánh giá khách quan, ổn định
            response_format={"type": "json_object"} # Cờ ép trả về JSON chuẩn
        )
        
        # Đọc chuỗi JSON trả về
        result_str = response.choices[0].message.content
        return json.loads(result_str)
    except Exception as e:
        print(f"Lỗi khi gọi Groq API: {e}")
        return None

# ==========================================
# VÒNG LẶP TỰ ĐỘNG XỬ LÝ TỪNG VĂN BẢN
# ==========================================
for index, row in df.iterrows():
    text_id = index + 1
    input_text = str(row.iloc[0]) 
    
    if pd.isna(input_text) or len(input_text.strip()) < 10:
        continue

    clear_output(wait=True)
    print(f"⏳ ĐANG XỬ LÝ VĂN BẢN SỐ: {text_id} / {len(df)} (Dòng {index + 2} trong Excel)")
    print(f"Tiến trình tổng: {round((text_id/len(df))*100, 1)}%")
    print("="*60)

    # 1. Reset môi trường data
    if os.path.exists(config.DATA_DIR):
        shutil.rmtree(config.DATA_DIR)
    os.makedirs(config.DATA_DIR)

    # 2. Chạy workflow sinh tóm tắt
    manager = ManagerAgent()
    try:
        final_output = manager.run_workflow(input_text=input_text, style="news_brief", resume=False)
        summary_result = final_output.get("summary", "")
    except Exception as e:
        summary_result = f"LỖI: {str(e)}"
        print(f"❌ Phát hiện lỗi tại văn bản {text_id}: {str(e)}")

    # 3. CHẤM ĐIỂM BẰNG GROQ API
    print("🤖 Đang gọi Groq API (Llama3-70B) để thẩm định chất lượng tóm tắt...")
    eval_result = evaluate_summary_with_groq(input_text, summary_result)
    
    # 4. ĐÓNG GÓI JSON KẾT QUẢ ĐẦU RA THEO ĐÚNG YÊU CẦU
    final_json_dict = {
        "id": text_id,
        "style": "news_brief",
        "article": input_text,
        "summary": summary_result,
        "faithfulness_score": "N/A",
        "faithfulness_reason": "Lỗi API",
        "relevance_score": "N/A",
        "relevance_reason": "Lỗi API",
        "coherence_conciseness_score": "N/A",
        "coherence_conciseness_reason": "Lỗi API"
    }

    # Nếu Groq trả về kết quả thành công, cập nhật đè lên các giá trị N/A
    if eval_result:
        final_json_dict.update({
            "faithfulness_score": eval_result.get("faithfulness_score", "N/A"),
            "faithfulness_reason": eval_result.get("faithfulness_reason", ""),
            "relevance_score": eval_result.get("relevance_score", "N/A"),
            "relevance_reason": eval_result.get("relevance_reason", ""),
            "coherence_conciseness_score": eval_result.get("coherence_conciseness_score", "N/A"),
            "coherence_conciseness_reason": eval_result.get("coherence_conciseness_reason", "")
        })

    # Chuyển Dict thành chuỗi định dạng JSON đẹp mắt, hỗ trợ tiếng Việt
    final_result_json = json.dumps(final_json_dict, ensure_ascii=False, indent=2)

    # 5. Ghi Excel và Lưu liên tục
    df.at[index, 'Summary'] = summary_result
    df.at[index, 'Result'] = final_result_json
    df.to_excel(OUTPUT_EXCEL_PATH, index=False)
    print(f"✅ Đã ghi thành công kết quả và cục JSON Đánh giá vào file Excel.")

    # 6. Sao lưu Session & Log
    backup_folder = os.path.join(BATCH_BACKUP_DIR, f"text_{text_id}")
    if os.path.exists(backup_folder):
        shutil.rmtree(backup_folder)
    shutil.copytree(config.DATA_DIR, backup_folder)

    # 7. DỌN DẸP BỘ NHỚ
    llm.unload()
    del manager      
    gc.collect()     

print("\n🎉 HOÀN TẤT! QUÁ TRÌNH TÓM TẮT & CHẤM ĐIỂM HÀNG LOẠT ĐÃ XONG!")